In [1]:
# Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
# Create Spark Session
spark = SparkSession.builder.appName("Spark Basics Assignment").getOrCreate()

## Load Dataset

In [3]:
# Load Dataset
df = spark.read.csv(
    "healthcare-dataset-stroke-data.csv",
    header=True,
    inferSchema=True
)
# View data
df.show(5)

# Column names
print(df.columns)

# Data types
df.printSchema()

+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|   id|gender| age|hypertension|heart_disease|ever_married|    work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
| 9046|  Male|67.0|           0|            1|         Yes|      Private|         Urban|           228.69|36.6|formerly smoked|     1|
|51676|Female|61.0|           0|            0|         Yes|Self-employed|         Rural|           202.21| N/A|   never smoked|     1|
|31112|  Male|80.0|           0|            1|         Yes|      Private|         Rural|           105.92|32.5|   never smoked|     1|
|60182|Female|49.0|           0|            0|         Yes|      Private|         Urban|           171.23|34.4|         smokes|     1|
| 1665|Female|79.0|           1|            0|         

## Data Cleaning

In [4]:
# Remove Duplicates
df = df.dropDuplicates()

In [5]:
# Checking missing values
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---+------+---+------------+-------------+------------+---------+--------------+-----------------+---+--------------+------+
| id|gender|age|hypertension|heart_disease|ever_married|work_type|Residence_type|avg_glucose_level|bmi|smoking_status|stroke|
+---+------+---+------------+-------------+------------+---------+--------------+-----------------+---+--------------+------+
|  0|     0|  0|           0|            0|           0|        0|             0|                0|  0|             0|     0|
+---+------+---+------------+-------------+------------+---------+--------------+-----------------+---+--------------+------+



In [6]:
# Fill missing BMI values with 0
df = df.na.fill({"bmi": 0})

# Fill missing smoking status with "Unknown"
df = df.na.fill({"smoking_status": "Unknown"})

## Filter Data

In [7]:
# Filter patients with age greater than 50
age_df = df.filter(col("age") > 50)
age_df.show()

+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|   id|gender| age|hypertension|heart_disease|ever_married|    work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|56669|  Male|81.0|           0|            0|         Yes|      Private|         Urban|           186.21|  29|formerly smoked|     1|
|49014|Female|76.0|           0|            0|         Yes|     Govt_job|         Urban|           204.05|23.5|   never smoked|     0|
|35022|Female|69.0|           0|            0|         Yes|      Private|         Urban|           111.48|  37|         smokes|     0|
|60586|Female|68.0|           0|            0|         Yes|      Private|         Rural|            85.29|27.1|formerly smoked|     0|
|52530|  Male|55.0|           0|            0|         

In [8]:
# Filter patients who work in the Private sector
private_df = df.filter(col("work_type") == "Private")
private_df.show()

+-----+------+----+------------+-------------+------------+---------+--------------+-----------------+----+---------------+------+
|   id|gender| age|hypertension|heart_disease|ever_married|work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+----+------------+-------------+------------+---------+--------------+-----------------+----+---------------+------+
|56669|  Male|81.0|           0|            0|         Yes|  Private|         Urban|           186.21|  29|formerly smoked|     1|
|31143|Female|22.0|           0|            0|          No|  Private|         Rural|           107.52|41.6|        Unknown|     0|
|33404|  Male|35.0|           0|            0|         Yes|  Private|         Urban|            89.32|36.7|        Unknown|     0|
|35022|Female|69.0|           0|            0|         Yes|  Private|         Urban|           111.48|  37|         smokes|     0|
|60586|Female|68.0|           0|            0|         Yes|  Private|         Rural

In [9]:
# Filter patients living in Urban areas
urban_df = df.filter(col("Residence_type") == "Urban")
urban_df.show()

+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|   id|gender| age|hypertension|heart_disease|ever_married|    work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|56669|  Male|81.0|           0|            0|         Yes|      Private|         Urban|           186.21|  29|formerly smoked|     1|
|33404|  Male|35.0|           0|            0|         Yes|      Private|         Urban|            89.32|36.7|        Unknown|     0|
|49014|Female|76.0|           0|            0|         Yes|     Govt_job|         Urban|           204.05|23.5|   never smoked|     0|
|35022|Female|69.0|           0|            0|         Yes|      Private|         Urban|           111.48|  37|         smokes|     0|
|  760|  Male| 0.8|           0|            0|         

## Data Transformation

In [10]:
# Rename avg_glucose_level column
df = df.withColumnRenamed("avg_glucose_level","glucose_level")

In [11]:
# Convert age to integer
df = df.withColumn("age",col("age").cast("integer"))

In [12]:
# Display updated schema
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- heart_disease: integer (nullable = true)
 |-- ever_married: string (nullable = true)
 |-- work_type: string (nullable = true)
 |-- Residence_type: string (nullable = true)
 |-- glucose_level: double (nullable = true)
 |-- bmi: string (nullable = false)
 |-- smoking_status: string (nullable = false)
 |-- stroke: integer (nullable = true)



## Aggregation

In [13]:
# Count total rows
print(df.count())

5110


In [14]:
# Average glucose level
df.select(avg("glucose_level").alias("Average Glucose")).show()

+------------------+
|   Average Glucose|
+------------------+
|106.14767710371808|
+------------------+



In [15]:
# Minimum glucose level
df.select(min("glucose_level").alias("Minimum Glucose")).show()

+---------------+
|Minimum Glucose|
+---------------+
|          55.12|
+---------------+



In [16]:
# Maximum glucose level
df.select(max("glucose_level").alias("Maximum Glucose")).show()

+---------------+
|Maximum Glucose|
+---------------+
|         271.74|
+---------------+



## Group By Operations

In [17]:
# Average glucose level by Residence Type
df.groupBy("Residence_type").avg("glucose_level").show()

+--------------+------------------+
|Residence_type|avg(glucose_level)|
+--------------+------------------+
|         Urban|105.92730739599389|
|         Rural|106.37523468575971|
+--------------+------------------+



In [18]:
# Count patients by Residence Type
df.groupBy("Residence_type").count().show()

+--------------+-----+
|Residence_type|count|
+--------------+-----+
|         Urban| 2596|
|         Rural| 2514|
+--------------+-----+



In [19]:
# Stroke count by Gender
df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female| 2994|
| Other|    1|
|  Male| 2115|
+------+-----+



## Final Processing Pipeline

In [21]:
result = (
    spark.read.csv(
        "healthcare-dataset-stroke-data.csv",
        header=True,
        inferSchema=True
    )
    .dropDuplicates()                 # Remove duplicate rows
    .na.fill({"bmi": 0})              # Fill missing BMI values
    .filter(col("age") > 50)          # Filter patients older than 50
    .withColumnRenamed(
        "avg_glucose_level",
        "glucose_level"
    )                                 # Rename column
    .groupBy("Residence_type")        # Group by residence type
    .agg(
        avg("glucose_level").alias("Average_Glucose")
    )
)

result.coalesce(1).write.mode("overwrite").option("header", True).csv("output/results")